In [3]:
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, LongType
from IPython.display import display

# Инициализация Spark-сессии
try:
    from lakehouse_spark import SparkSessionContext
    # Если у тебя есть контекст платформы, используем его
    ctx = SparkSessionContext("case-01-stage-clean")
    spark = ctx.__enter__() # Входим в контекст, чтобы spark стал доступен
    print("Spark-сессия успешно создана через lakehouse_spark!")
except ImportError:
    # Если это обычный Jupyter без платформы
    from pyspark.sql import SparkSession
    spark = SparkSession.builder \
        .appName("Case01_ETL") \
        .getOrCreate()
    print("Spark-сессия создана через стандартный PySpark!")


def log_funnel(table_label: str, step_label: str, reason: str, before_cnt: int, after_cnt: int):
    removed = before_cnt - after_cnt
    pct = round((removed / before_cnt * 100), 2) if before_cnt > 0 else 0.0
    funnel_records.append({
        "Таблица": table_label,
        "Этап очистки": step_label,
        "Причина урезания": reason,
        "Было строк": before_cnt,
        "Стало строк": after_cnt,
        "Урезано строк": removed,
        "Урезано (%)": pct
    })

# Инициализация Spark (если у тебя нет контекста lakehouse_spark, используй свой)
# with SparkSessionContext("case-01-stage-clean") as ctx:
#     spark = ctx
# Для Jupyter используем глобальный spark

# -------------------------------------------------------------------------
# 1. ОЧИСТКА МАТЧЕЙ (clean_matches)
# -------------------------------------------------------------------------
raw_m = spark.table("lakehouse.raw.pg__matches")
c0 = raw_m.count()

w_match = Window.partitionBy("match_id").orderBy(F.col("sys_insert_stamp").desc())
m_dedup = raw_m.withColumn("rn", F.row_number().over(w_match)).filter("rn = 1").drop("rn")
c1 = m_dedup.count()
log_funnel("matches", "1. Дедупликация", "Повторные записи по match_id", c0, c1)

m_valid = m_dedup.filter("human_players = 10 AND duration >= 600")
c2 = m_valid.count()
log_funnel("matches", "2. Валидация", "Матчи с ливерами (!=10 игр.) или ранние ливы (<10 мин)", c1, c2)

clean_matches = m_valid.select("match_id").cache()

# -------------------------------------------------------------------------
# 2. ОЧИСТКА ИГРОКОВ В МАТЧАХ (clean_match_players)
# -------------------------------------------------------------------------
raw_p = spark.table("lakehouse.raw.pg__match_players")
p0 = raw_p.count()

w_player = Window.partitionBy("match_id", "player_slot").orderBy(F.col("sys_insert_stamp").desc())
p_dedup = raw_p.withColumn("rn", F.row_number().over(w_player)).filter("rn = 1").drop("rn")
p1 = p_dedup.count()
log_funnel("match_players", "1. Дедупликация", "Дубли по ключу (match_id, player_slot)", p0, p1)

# Убираем скрытые профили (4294967295) и пустые значения
p_valid = p_dedup.filter("account_id IS NOT NULL AND account_id <> 4294967295")
p2 = p_valid.count()
log_funnel("match_players", "2. Удаление анонимов", "Скрытые профили (account_id = 4294967295) или NULL", p1, p2)

# Оставляем только игроков из валидных матчей
p_in_matches = p_valid.join(clean_matches, "match_id", "inner")
p3 = p_in_matches.count()
log_funnel("match_players", "3. Ссылочная целостность", "Игроки из отсеянных матчей", p2, p3)

clean_players = p_in_matches.select(
    F.col("match_id").cast(LongType()),
    F.col("player_slot").cast(IntegerType()),
    F.col("account_id").cast(LongType()),
    F.col("hero_id").cast(IntegerType()),
    F.col("win").cast("boolean"),
    F.col("kills").cast(IntegerType()),
    F.col("deaths").cast(IntegerType()),
    F.col("assists").cast(IntegerType()),
    F.col("gold_per_min").cast(IntegerType()),
    F.col("xp_per_min").cast(IntegerType()),
    F.col("last_hits").cast(IntegerType()),
    F.when(F.col("player_slot") < 128, True).otherwise(False).alias("is_radiant")
).cache()

# -------------------------------------------------------------------------
# 3. ОЧИСТКА НИКОВ (clean_nicks)
# -------------------------------------------------------------------------
raw_n = spark.table("lakehouse.raw.pg__players")
n0 = raw_n.count()

w_nick = Window.partitionBy("account_id").orderBy(F.col("sys_insert_stamp").desc())
n_dedup = raw_n.withColumn("rn", F.row_number().over(w_nick)).filter("rn = 1").drop("rn")
n1 = n_dedup.count()
log_funnel("players", "1. Дедупликация", "Повторные записи по account_id", n0, n1)

n_valid = n_dedup.filter("personaname IS NOT NULL AND TRIM(personaname) <> ''")
n2 = n_valid.count()
log_funnel("players", "2. Удаление пустых", "Ники, состоящие из пробелов или NULL", n1, n2)

clean_nicks = n_valid.select("account_id", F.trim("personaname").alias("personaname")).cache()

# -------------------------------------------------------------------------
# 4. АГРЕГАЦИЯ ДЛЯ LLM (Только математика, без бизнес-логики)
# -------------------------------------------------------------------------
# Считаем статистику по каждому игроку (5+ матчей), чтобы передать ее в LLM
player_stats = clean_players.groupBy("account_id").agg(
    F.count("match_id").alias("total_matches"),
    F.avg(F.col("win").cast("int")).alias("winrate"),
    # KDA с защитой от деления на ноль
    F.avg((F.col("kills") + F.col("assists")) / F.greatest(F.col("deaths"), F.lit(1))).alias("avg_kda"),
    F.avg("gold_per_min").alias("avg_gpm"),
    F.avg("xp_per_min").alias("avg_xpm"),
    F.avg("last_hits").alias("avg_last_hits"),
    F.countDistinct("hero_id").alias("distinct_heroes_count")
).filter("total_matches >= 5")

# Собираем итоговую STAGE-таблицу (одна строка = один игрок)
final_stage = clean_nicks.alias("n") \
    .join(player_stats.alias("s"), "account_id", "inner") \
    .select(
        "n.account_id",
        "n.personaname",
        "s.total_matches",
        F.round("s.winrate", 2).alias("winrate"),
        F.round("s.avg_kda", 2).alias("avg_kda"),
        F.round("s.avg_gpm", 0).alias("avg_gpm"),
        F.round("s.avg_xpm", 0).alias("avg_xpm"),
        F.round("s.avg_last_hits", 1).alias("avg_last_hits"),
        "s.distinct_heroes_count"
    )

# Сохранение (опционально, если нужно писать в Iceberg)
# final_stage.writeTo("lakehouse.stage.practice_case_01_clean_players").using("iceberg").createOrReplace()

# -------------------------------------------------------------------------
# 5. ВЫВОД ВОРОНКИ И РЕЗУЛЬТАТА
# -------------------------------------------------------------------------
print("=== ВОРОНКА ОЧИСТКИ ДАННЫХ (STAGE): ГДЕ И ПОЧЕМУ УРЕЗАНЫ ЗНАЧЕНИЯ ===")
funnel_df = pd.DataFrame(funnel_records)
display(funnel_df)

print(f"\nИтого игроков, готовых к отправке в LLM (>=5 матчей): {final_stage.count()}")
print("=== Пример подготовленных данных ===")

# Вариант вывода через Pandas (для красивого отображения в Jupyter)
display(final_stage.limit(5).toPandas())

# Альтернативный вариант (если display так и не заработает) - просто Spark-вывод:
# final_stage.show(5, truncate=False)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/14 13:55:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark-сессия успешно создана через lakehouse_spark!


=== ВОРОНКА ОЧИСТКИ ДАННЫХ (STAGE): ГДЕ И ПОЧЕМУ УРЕЗАНЫ ЗНАЧЕНИЯ ===


,Таблица,Этап очистки,Причина урезания,Было строк,Стало строк,Урезано строк,Урезано (%)
0,matches,1. Дедупликация,Повторные записи по match_id,322742,322742,0,0.00
1,matches,2. Валидация,Матчи с ливерами (!=10 игр.) или ранние ливы (...,322742,322377,365,0.11
2,match_players,1. Дедупликация,"Дубли по ключу (match_id, player_slot)",4855502,4855502,0,0.00
3,match_players,2. Удаление анонимов,Скрытые профили (account_id = 4294967295) или ...,4855502,3265286,1590216,32.75
4,match_players,3. Ссылочная целостность,Игроки из отсеянных матчей,3265286,2585167,680119,20.83
5,players,1. Дедупликация,Повторные записи по account_id,1763695,1763695,0,0.00
6,players,2. Удаление пустых,"Ники, состоящие из пробелов или NULL",1763695,473411,1290284,73.16



Итого игроков, готовых к отправке в LLM (>=5 матчей): 18159
=== Пример подготовленных данных ===


,account_id,personaname,total_matches,winrate,avg_kda,avg_gpm,avg_xpm,avg_last_hits,distinct_heroes_count
0,100788,Vaan,5,0.00,1.91,766.0,1388.0,78.4,4
1,270788,LuGang,6,0.50,2.07,1098.0,2051.0,157.7,6
2,287478,uni',11,0.36,7.48,483.0,635.0,160.1,6
3,413850,YourSoul1sMine,7,0.71,3.84,462.0,661.0,170.0,7
4,994063,РедкийХуй,9,0.11,2.40,438.0,646.0,165.9,9
